# 🛠️ 0. Workspace & Environment Setup Template

본 노트북은 에이전트 개발 및 실습을 위한 **표준 환경 설정 템플릿**입니다.

프로젝트 루트 경로 탐색, 환경변수(`.env`) 로드, 비동기 이벤트 루프(`nest_asyncio`) 설정 및 브라우저 환경변수 제어를 포함합니다.

---

### 📋 포함된 설정 항목
1. **환경변수 로드**: `.env` 파일 로드 (`load_dotenv`)
2. **프로젝트 루트 경로 자동 탐색**: `app/` 디렉터리를 기준으로 루트 경로 감지 및 `sys.path` 등록
3. **비동기 이벤트 루프 설정**: Jupyter 환경에서 `asyncio` 중복 실행 방지 (`nest_asyncio`)
4. **브라우저 실행 모드 제어**: `HEADLESS` 설정 (`true` / `false`)
5. **공통 유틸리티 확인**: `normalize_content`, `init_chat_model` 등 기본 모듈 테스트

## 🛠️ Step 0. 환경 세팅

필요한 기본 패키지를 로드하고 프로젝트 루트 경로 및 비동기 이벤트 루프(`nest_asyncio`)를 설정합니다.

In [ ]:
import os
import sys
import asyncio
import nest_asyncio
from dotenv import load_dotenv

# 1. 주피터 노트북 비동기 루프 중복 방지
nest_asyncio.apply()

# 2. 프로젝트 루트 상향 동적 탐색 (어느 서브 폴더에 노트북이 있어도 100% 작동)
def find_project_root():
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.exists(os.path.join(p, "app")) and (
            os.path.exists(os.path.join(p, ".env")) or os.path.exists(os.path.join(p, "configs"))
        ):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.path.join(os.getcwd(), ".."))

project_root = find_project_root()
os.chdir(project_root)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 3. 환경변수 명시적 로드
dotenv_path = os.path.join(project_root, ".env")
load_dotenv(dotenv_path, override=True)

print(f"✅ Working Directory: {os.getcwd()}")
print(f"✅ Project Root: {project_root}")

# 4. 기본 공통 유틸리티 임포트
from app.utils import normalize_content

## 🌐 Step 1. 브라우저 및 실행 옵션 설정

브라우저 기반 탐색 도구(Playwright, Navigator, browser-use) 사용 시 창 표시 여부를 설정합니다.
- `HEADLESS = "false"`: 브라우저 GUI 창을 띄워 실시간 동작 확인 (로컬 또는 noVNC)
- `HEADLESS = "true"`: 화면 없이 백그라운드에서 가볍게 실행 (기본값)

In [ ]:
# 브라우저 표시 모드 설정 (false: 브라우저 창 표시, true: 백그라운드 실행)
os.environ['HEADLESS'] = "false"
print(f"🌐 HEADLESS mode: {os.environ.get('HEADLESS', 'true')}")

## 🧪 Step 2. 기본 LLM 초기화 및 환경 테스트

설정된 환경변수(`OPENAI_API_KEY`, `ANTHROPIC_API_KEY` 등)를 바탕으로 기본 챗 모델 인스턴스 생성을 확인합니다.

In [ ]:
from app.utils import init_chat_model

try:
    llm = init_chat_model()
    print(f"✅ 기본 LLM 모델 로드 성공: {llm}")
except Exception as e:
    print(f"⚠️ LLM 초기화 확인 (API Key 설정 확인 필요): {e}")